# T4 verification (Colab / Kaggle)

Runtime -> Change runtime type -> **T4 GPU**. Checks the open eval-machine risks on real hardware:

1. the pipeline picks **CUDA** (device logic: `src/perception/detector.py::resolve_device`, cuda > mps > cpu);
2. **fp16** works for YOLO *and* Qwen3-VL (T4 has no bf16; a NaN would silently reject every VLM candidate -- the code now logs it as an error);
3. wall-clock time of `run_submission.py` on one video vs the 3x budget (target 1.5x), per stage;
4. `weights/download.sh` then an offline-style run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| bf16", torch.cuda.is_available() and torch.cuda.is_bf16_supported())
assert torch.cuda.is_available(), "Pick a T4 GPU runtime first"

## 1. Repo + dependencies (the repo must be public, or clone with a token)

In [ ]:
!rm -rf wiut && git clone -q https://github.com/Bilolceo/Wiut-.git wiut
%cd wiut
!git log --oneline -1
!pip install -q -r requirements.txt
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 2. Weights -- exactly what the organizers run once, with internet

In [ ]:
!bash weights/download.sh

## 3. One short sample video (C3905, 2 min) from your Drive copy

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import glob, os, shutil
src = [p for p in glob.glob("/content/drive/MyDrive/**/*C3905*.MP4", recursive=True)]
print(src)
os.makedirs("/content/one_video", exist_ok=True)
shutil.copy(src[0], "/content/one_video/C3905.MP4")   # local copy: Drive I/O would distort the timing
VIDEO_DIR = "/content/one_video"

## 4. Device + fp16 sanity for both models

In [ ]:
import time, sys, numpy as np
sys.path.insert(0, ".")
from src.perception.detector import Perception, resolve_device
from src.detect import get_vlm_backend
from src.verify.vlm import read_clip, ClipSpec, SUFFIX

print("resolve_device('auto') ->", resolve_device("auto"))
p = Perception(model_path="weights/yolo11n.pt", tracker="configs/tracker_botsort.yaml")
print("YOLO device", p.device, "| half", p.half)

t = time.perf_counter(); vlm = get_vlm_backend("weights/qwen3-vl-2b-instruct")
print(f"VLM load {time.perf_counter()-t:.1f}s | device {vlm.device} | dtype {next(vlm.model.parameters()).dtype}")
frames = read_clip(f"{VIDEO_DIR}/C3905.MP4", 40, 41, ClipSpec(n_frames=4))
for q in ["Is there a bus in these frames?", "Is there an elephant in these frames?"]:
    t = time.perf_counter(); pr = vlm.p_yes(frames, q + SUFFIX)
    print(f"{q:40s} p_yes={pr:.3f}  finite={np.isfinite(pr)}  {time.perf_counter()-t:.2f}s")
# expected: bus ~1.0, elephant ~0.0, both finite. NaN here = fp16 overflow -> report it.

## 5. Part A per stage (one video)

In [ ]:
from src.detect import detect_events_impl
torch.cuda.reset_peak_memory_stats()
events, rep = detect_events_impl(f"{VIDEO_DIR}/C3905.MP4")
print({k: v for k, v in rep.items() if k.startswith("sec_") or k in ("duration_sec", "runtime_factor", "n_tracks")})
print(len(events), "events; peak VRAM %.1f GB" % (torch.cuda.max_memory_allocated() / 1e9))

## 6. The official harness, timed (Part A + B, must be <= 3x duration; target 1.5x)

In [ ]:
!python run_submission.py --videos "$VIDEO_DIR" --out predictions_t4.json --team t4-check
!python evaluate.py --pred predictions_t4.json --validate-only
import json; log = json.load(open("predictions_t4.json"))["log"]["C3905.MP4"]
print(log, "-> factor %.2fx" % (log["total_sec"] / log["duration"]))

## Report back
`resolve_device`, YOLO half, VLM dtype + the two p_yes values, the stage timings from step 5, peak VRAM, and the factor from step 6.
If step 6 is above ~1.5x, the first lever is `verifier.max_calls` in `configs/runtime.yaml`.